# Extract a Steering Vector

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/modulabs-personalab/psyctl/blob/main/examples/en/04_extract_vector.ipynb)

This notebook walks through extracting steering vectors from a contrastive dataset using PSYCTL's three extraction methods:

1. **Mean Difference** (`mean_diff`) — Fast, simple average of activation differences
2. **Denoised Mean Difference** (`denoised_mean_diff`) — PCA-based denoising for noise reduction
3. **BiPO** (`bipo`) — Gradient-based optimization using DPO loss (best quality)

**In this notebook you will:**
1. Load a steering dataset from HuggingFace
2. Extract vectors with all three methods
3. Compare responses side-by-side
4. Save and optionally share results

**Requirements:** Colab free tier (T4 GPU). [HuggingFace token](https://huggingface.co/settings/tokens).

**Time:** ~10 minutes

In [ ]:
!pip install -q git+https://github.com/modulabs-personalab/psyctl.git bitsandbytes accelerate datasets

In [ ]:
import os

try:
    from google.colab import userdata
    _hf = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = _hf if isinstance(_hf, str) else str(_hf)
    print("Loaded HF_TOKEN from Colab Secrets")
except Exception:
    if not os.environ.get("HF_TOKEN"):
        raise RuntimeError(
            "HF_TOKEN not found. Please:\n"
            "1. Click the key icon (Secrets) in the left sidebar\n"
            "2. Add HF_TOKEN with your HuggingFace token\n"
            "3. Toggle 'Notebook access' ON for HF_TOKEN\n"
            "4. Re-run this cell"
        )

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PSYCTL_LOG_LEVEL"] = "WARNING"

## Load Dataset

We use a community-contributed dataset from HuggingFace. You can also use a local JSONL file generated with [03_generate_dataset.ipynb](./03_generate_dataset.ipynb).

In [ ]:
from pathlib import Path
from datasets import load_dataset

# Download a community steering dataset
DATASET_NAME = "CaveduckAI/steer-personality-extroversion-ko"
ds = load_dataset(DATASET_NAME, split="train")

# Save as local JSONL for PSYCTL
dataset_dir = Path("./dataset_demo")
dataset_dir.mkdir(exist_ok=True)
dataset_path = dataset_dir / "data.jsonl"
ds.to_json(str(dataset_path))

print(f"Dataset: {DATASET_NAME}")
print(f"Samples: {len(ds)}")
print(f"Saved to: {dataset_path}")
print(f"\nSample keys: {list(ds[0].keys())}")

## Method 1: Mean Difference (Fastest)

Computes the mean activation difference between positive and neutral responses. This is the simplest and fastest method from the [CAA paper](https://arxiv.org/abs/2312.06681).

In [ ]:
from psyctl.core.steering_extractor import SteeringExtractor

SMALL_MODEL = "google/gemma-3-270m-it"
TARGET_LAYER = "model.layers.9.mlp.down_proj"  # Middle layer of 18-layer model

extractor = SteeringExtractor()

output_mean_diff = Path("./vectors/mean_diff.safetensors")
vectors_md = extractor.extract_steering_vector(
    model_name=SMALL_MODEL,
    layers=[TARGET_LAYER],
    dataset_path=dataset_path,
    output_path=output_mean_diff,
    method="mean_diff",
    normalize=True,
)
print(f"Mean diff vector shape: {list(vectors_md.values())[0].shape}")
print(f"Saved to: {output_mean_diff}")

## Method 2: Denoised Mean Difference (PCA-Based)

Applies PCA denoising to reduce noise in the activation differences before averaging. More robust than plain mean diff.

In [ ]:
output_denoised = Path("./vectors/denoised_mean_diff.safetensors")
vectors_dmd = extractor.extract_steering_vector(
    model_name=SMALL_MODEL,
    layers=[TARGET_LAYER],
    dataset_path=dataset_path,
    output_path=output_denoised,
    method="denoised_mean_diff",
    normalize=True,
    variance_threshold=0.95,
)
print(f"Denoised mean diff vector shape: {list(vectors_dmd.values())[0].shape}")
print(f"Saved to: {output_denoised}")

## Method 3: BiPO (Gradient-Based, Best Quality)

Uses [Bidirectional Preference Optimization](https://arxiv.org/abs/2406.00045) — a gradient-based method that optimizes the steering vector using DPO loss. Slower but typically produces the best steering quality.

In [ ]:
output_bipo = Path("./vectors/bipo.safetensors")
vectors_bipo = extractor.extract_steering_vector(
    model_name=SMALL_MODEL,
    layers=[TARGET_LAYER],
    dataset_path=dataset_path,
    output_path=output_bipo,
    method="bipo",
    normalize=True,
    epochs=3,
    lr=1e-3,
    use_chat_template=False,
)
print(f"BiPO vector shape: {list(vectors_bipo.values())[0].shape}")
print(f"Saved to: {output_bipo}")

## Compare All Three Methods

Apply each vector to the same prompt and compare the steered outputs.

In [ ]:
from psyctl.core.steering_applier import SteeringApplier

applier = SteeringApplier()
prompt = "Hello, how are you?"

methods = {
    "mean_diff": output_mean_diff,
    "denoised_mean_diff": output_denoised,
    "bipo": output_bipo,
}

print(f"Prompt: \"{prompt}\"")
print("=" * 70)

# Baseline (no steering)
print("\n--- Baseline ---")
print(applier.apply_steering(
    model_name=SMALL_MODEL,
    steering_vector_path=output_mean_diff,
    input_text=prompt,
    strength=0.0,
    max_new_tokens=100,
    temperature=0.7,
))

for method_name, vpath in methods.items():
    print(f"\n--- {method_name} (strength=1.5) ---")
    print(applier.apply_steering(
        model_name=SMALL_MODEL,
        steering_vector_path=vpath,
        input_text=prompt,
        strength=1.5,
        max_new_tokens=100,
        temperature=0.7,
    ))

## Save and Share

Your extracted vectors are saved as `.safetensors` files in `./vectors/`. You can upload them to HuggingFace Hub to share with the community:

```python
from huggingface_hub import HfApi
api = HfApi()
api.upload_file(
    path_or_fileobj="./vectors/bipo.safetensors",
    path_in_repo="my_vector.safetensors",
    repo_id="your-username/my-steering-vectors",
    repo_type="model",
)
```

**Next:** Try [05_layer_analysis.ipynb](./05_layer_analysis.ipynb) to find the optimal layers for extraction.